# Hybrid VTG - Google Colab Benchmark Runner

This notebook allows you to easily run **Hybrid VTG** (training-free video temporal grounding benchmarks) on Google Colab using GPU acceleration.

### Step-by-Step Overview:
1. **Check GPU environment** (`nvidia-smi`)
2. **Mount Google Drive** (To save datasets permanently!)
3. **Clone & Install repository**
4. **Download datasets and model checkpoints**
5. **Run benchmark evaluation**
6. **Display results**

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (Highly Recommended)
Google Colab deletes all files when your session disconnects. By mounting Google Drive and symlinking our `assets` folder, you **only have to download the 3.5GB datasets once**. 

On future runs, the data will instantly be available.

In [ ]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Create a permanent assets folder in your Google Drive
DRIVE_ASSETS = "/content/drive/MyDrive/hybrid_vtg_assets"
os.makedirs(DRIVE_ASSETS, exist_ok=True)
print(f"\nAssets will be permanently saved to: {DRIVE_ASSETS}")

## 3. Repository Setup & Installation

*Note: Update `GITHUB_URL` with your GitHub repository URL if cloning into a fresh Colab instance.*

In [ ]:
import os

GITHUB_URL = "https://github.com/YOUR_USERNAME/hybrid_vtg.git"  # <--- Replace with your GitHub repo URL

if os.path.exists("pyproject.toml"):
    print("Already in repository root:", os.getcwd())
elif os.path.exists("hybrid_vtg/pyproject.toml"):
    %cd hybrid_vtg
    print("Switched directory to repo root:", os.getcwd())
else:
    !git clone {GITHUB_URL}
    %cd hybrid_vtg
    print("Cloned and switched to repo root:", os.getcwd())

# Symlink the Google Drive assets folder into the repository
if not os.path.exists("./assets"):
    os.symlink(DRIVE_ASSETS, "./assets")
    print("Symlinked ./assets -> Google Drive. Datasets are now persistent!")

In [ ]:
# Install dependencies cleanly into Colab environment
%pip install -q "opencv-python-headless>=4.9,<5" "numpy>=1.26,<2.1"
%pip install -e '.[downloads,test]'

# Add src/ directory to Python path for current session
import sys
if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))

## 4. Download Benchmarks & Checkpoints

Download required benchmark assets and pretrained checkpoints into `./assets`.

Available target options:
- `omtg` (OMTG Bench)
- `tacos` (TACoS test split)
- `qvhighlights` (QVHighlights test split)
- `timelens2-4b` (TimeLens2 4B model)
- `univtg` (UniVTG checkpoint)

*Because we symlinked `./assets` to Google Drive above, this download will save directly to your Drive!*

In [ ]:
# Download assets using PYTHONPATH=src
!PYTHONPATH=src python -m hybrid_vtg.cli download omtg timelens2-4b --root ./assets --accept-licenses

## 5. Run Benchmark Execution

Select benchmark, model backend, method, subset percentage, and random seed.

In [ ]:
# Run evaluation
!PYTHONPATH=src python -m hybrid_vtg.cli run \
  --benchmark omtg \
  --model qwen3-vl-4b \
  --method coarse-to-fine-64 \
  --subset 10 \
  --seed 42

## 6. View Evaluation Results

Display generated `RESULTS.md` directly in the notebook.

In [ ]:
from IPython.display import Markdown, display
import os

results_path = "results/RESULTS.md"
if os.path.exists(results_path):
    with open(results_path, "r") as f:
        display(Markdown(f.read()))
else:
    print("RESULTS.md not found. Make sure a benchmark run has completed.")